# Student Performance & At-Risk Prediction

## Objective

Generate a reproducible synthetic educational dataset
for predicting students who may require additional
academic support.

## Dataset Specifications

- Students: 5,000
- Historical assessments: 3
- Prediction period: January 2025 – October 2025
- Prediction target: Next assessment score below 40
- Prediction type: Binary Classification

## Dataset Features

1. Attendance
2. Recent Attendance
3. Assignment Completion
4. Missing Assignments
5. Late Submission Rate
6. Previous Assessment Scores
7. Submission Recency

## Responsible AI

The dataset is completely synthetic.

No real student information is used.

The model will provide academic support recommendations,
not automated academic decisions.

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

In [2]:
# Dataset configuration

n_students = 5000
random_seed = 42
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-10-31")
risk_threshold = 40
min_assestments = 15
max_assestments = 25
# Random number generator
rng = np.random.default_rng(random_seed)

In [3]:
print("Total Students:", n_students)

print("Prediction Period:", start_date, "to", end_date)

print("At-Risk Threshold:", risk_threshold)

print("Random Seed:", random_seed)

Total Students: 5000
Prediction Period: 2025-01-01 00:00:00 to 2025-10-31 00:00:00
At-Risk Threshold: 40
Random Seed: 42


In [4]:
student_ids = np.array([f"STU{i:05d}"for i in range(1, n_students + 1)])
total_days = (end_date - start_date).days
date_offsets = rng.integers(0,total_days + 1,size=n_students)
prediction_dates = start_date + pd.to_timedelta(date_offsets,unit="D")
cohort_ids = prediction_dates.to_period("Q").astype(str)

In [5]:
student_base = pd.DataFrame({
    "student_id": student_ids,
    "cohort_id": cohort_ids,
    "prediction_date": prediction_dates
})

In [6]:
display(student_base.head(10))

,student_id,cohort_id,prediction_date
0,STU00001,2025Q1,2025-01-28
1,STU00002,2025Q3,2025-08-24
2,STU00003,2025Q3,2025-07-18
3,STU00004,2025Q2,2025-05-14
4,STU00005,2025Q2,2025-05-12
5,STU00006,2025Q3,2025-09-19
6,STU00007,2025Q1,2025-01-27
7,STU00008,2025Q3,2025-07-31
8,STU00009,2025Q1,2025-03-03
9,STU00010,2025Q1,2025-01-29


In [7]:
print("Total Records:", len(student_base))
print("Unique Students:",student_base["student_id"].nunique())
print("Prediction Date Range:",student_base["prediction_date"].min(),"to",student_base["prediction_date"].max())

Total Records: 5000
Unique Students: 5000
Prediction Date Range: 2025-01-01 00:00:00 to 2025-10-31 00:00:00


In [8]:
display(
    student_base["cohort_id"]
    .value_counts()
    .sort_index()
)

cohort_id
2025Q1    1489
2025Q2    1521
2025Q3    1462
2025Q4     528
Name: count, dtype: int64

In [9]:
assert len(student_base) == n_students
assert student_base["student_id"].is_unique
assert student_base["prediction_date"].between(start_date,end_date).all()
print("Student identity and date validation passed.")

Student identity and date validation passed.


In [10]:
engagement = rng.beta(a=2.0,b=2.2,size=n_students)

In [11]:
independent_strength = rng.beta(a=2.2,b=2.2,size=n_students)

In [12]:
academic_strength = (0.65 * engagement+ 0.35 * independent_strength)
academic_strength = np.clip(academic_strength,0,1)

In [13]:
latent_summary = pd.DataFrame({
    "engagement": engagement,
    "academic_strength": academic_strength
})
display(latent_summary.describe().round(3))

,engagement,academic_strength
count,5000.000,5000.000
mean,0.476,0.485
std,0.218,0.164
min,0.004,0.024
25%,0.309,0.364
50%,0.466,0.481
75%,0.643,0.606
max,0.997,0.941


In [14]:
assert np.all((engagement >= 0) & (engagement <= 1))
assert np.all(
    (academic_strength >= 0) &
    (academic_strength <= 1)
)
print("Academic characteristic generation passed.")

Academic characteristic generation passed.


In [15]:
# Generate overall attendance

attendance_noise = rng.normal(
    loc=0,
    scale=7,
    size=n_students
)
attendance_rate = (45+ 50 * engagement+ attendance_noise)
attendance_rate = np.clip(attendance_rate,25,100)

In [16]:
# Generate recent attendance with some variation

recent_attendance_noise = rng.normal(loc=0,scale=8,size=n_students)
recent_attendance_rate = np.clip(attendance_rate + recent_attendance_noise,0,100)

In [17]:
attendance_summary = pd.DataFrame({
    "attendance_rate": attendance_rate,
    "recent_attendance_rate": recent_attendance_rate
})
display(attendance_summary.describe().round(2))

,attendance_rate,recent_attendance_rate
count,5000.00,5000.00
mean,68.75,68.58
std,12.96,15.15
min,29.10,16.67
25%,59.20,58.22
50%,68.59,68.22
75%,78.27,79.51
max,100.00,100.00


In [18]:
assert np.all(
    (attendance_rate >= 0) &
    (attendance_rate <= 100)
)
assert np.all(
    (recent_attendance_rate >= 0) &
    (recent_attendance_rate <= 100)
)
print("Attendance validation passed.")

Attendance validation passed.


In [19]:
assignments_due = rng.integers(min_assestments,max_assestments + 1,size=n_students)

print("Minimum assignments:", assignments_due.min())
print("Maximum assignments:", assignments_due.max())
print("Average assignments:", assignments_due.mean())

Minimum assignments: 15
Maximum assignments: 25
Average assignments: 20.011


In [20]:
completion_probability = (0.35 + 0.58 * engagement)
completion_probability = np.clip(completion_probability,0.15,0.98)

In [21]:
print(pd.Series(completion_probability).describe().round(3))

count    5000.000
mean        0.626
std         0.127
min         0.352
25%         0.529
50%         0.621
75%         0.723
max         0.928
dtype: float64


In [22]:
assignments_completed = rng.binomial(n=assignments_due,p=completion_probability)
missing_assignments = (assignments_due - assignments_completed)
assignment_completion_rate = (assignments_completed / assignments_due) * 100

In [23]:
assignment_summary = pd.DataFrame({
    "assignments_due": assignments_due,
    "assignments_completed": assignments_completed,
    "missing_assignments": missing_assignments,
    "assignment_completion_rate": assignment_completion_rate
})
display(assignment_summary.head(10))

,assignments_due,assignments_completed,missing_assignments,assignment_completion_rate
0,19,10,9,52.631579
1,23,13,10,56.521739
2,20,10,10,50.000000
3,25,14,11,56.000000
4,22,10,12,45.454545
5,25,11,14,44.000000
6,25,12,13,48.000000
7,17,14,3,82.352941
8,18,14,4,77.777778
9,18,10,8,55.555556


In [24]:
display(assignment_summary.describe().round(2))

,assignments_due,assignments_completed,missing_assignments,assignment_completion_rate
count,5000.00,5000.00,5000.00,5000.00
mean,20.01,12.47,7.54,62.24
std,3.20,3.95,3.53,16.77
min,15.00,2.00,0.00,9.09
25%,17.00,10.00,5.00,50.00
50%,20.00,12.00,7.00,62.50
75%,23.00,15.00,10.00,75.00
max,25.00,25.00,20.00,100.00


In [25]:
late_submission_probability = (0.45 - 0.32 * engagement)
late_submission_probability = np.clip(late_submission_probability,0.05,0.50)
late_submissions = rng.binomial(
    n=assignments_completed,
    p=late_submission_probability
)
late_submission_rate = np.divide(late_submissions,assignments_completed,out=np.zeros(n_students, dtype=float),where=assignments_completed > 0) * 100

In [26]:
submission_recency = (2+ 14 * (1 - engagement)+ rng.normal(0, 3, n_students))
days_since_last_submission = np.clip(np.rint(submission_recency),0,30).astype(int)

# Students with no completed assignments
# are treated as having no recent submission.
days_since_last_submission[assignments_completed == 0] = 30

In [27]:
assignment_features = pd.DataFrame({
    "assignment_completion_rate": assignment_completion_rate,
    "missing_assignments": missing_assignments,
    "late_submission_rate": late_submission_rate,
    "days_since_last_submission": days_since_last_submission
})
display(assignment_features.describe().round(2))

,assignment_completion_rate,missing_assignments,late_submission_rate,days_since_last_submission
count,5000.00,5000.00,5000.00,5000.00
mean,62.24,7.54,29.93,9.38
std,16.77,3.53,15.35,4.19
min,9.09,0.00,0.00,0.00
25%,50.00,5.00,18.75,6.00
50%,62.50,7.00,28.57,9.00
75%,75.00,10.00,38.89,12.00
max,100.00,20.00,100.00,23.00


In [28]:
student_data = student_base.copy()
student_data["attendance_rate"] = attendance_rate
student_data["recent_attendance_rate"] = recent_attendance_rate
student_data["assignments_due"] = assignments_due
student_data["assignments_completed"] = assignments_completed
student_data["assignment_completion_rate"] = assignment_completion_rate
student_data["missing_assignments"] = missing_assignments
student_data["late_submissions"] = late_submissions
student_data["late_submission_rate"] = late_submission_rate
student_data["days_since_last_submission"] = days_since_last_submission

In [29]:
display(student_data.head(10))

,student_id,cohort_id,prediction_date,attendance_rate,recent_attendance_rate,assignments_due,assignments_completed,assignment_completion_rate,missing_assignments,late_submissions,late_submission_rate,days_since_last_submission
0,STU00001,2025Q1,2025-01-28,65.595820,62.323629,19,10,52.631579,9,3,30.000000,12
1,STU00002,2025Q3,2025-08-24,59.001961,55.013889,23,13,56.521739,10,6,46.153846,16
2,STU00003,2025Q3,2025-07-18,53.932610,55.796970,20,10,50.000000,10,6,60.000000,10
3,STU00004,2025Q2,2025-05-14,55.484107,65.226436,25,14,56.000000,11,3,21.428571,15
4,STU00005,2025Q2,2025-05-12,81.939543,72.299291,22,10,45.454545,12,2,20.000000,14
5,STU00006,2025Q3,2025-09-19,63.302305,59.459912,25,11,44.000000,14,3,27.272727,8
6,STU00007,2025Q1,2025-01-27,62.977834,64.279166,25,12,48.000000,13,7,58.333333,6
7,STU00008,2025Q3,2025-07-31,74.711687,88.766565,17,14,82.352941,3,3,21.428571,6
8,STU00009,2025Q1,2025-03-03,63.554742,67.008276,18,14,77.777778,4,4,28.571429,13
9,STU00010,2025Q1,2025-01-29,70.069226,63.567587,18,10,55.555556,8,4,40.000000,9


In [30]:
print("Dataset shape:", student_data.shape)
print("\nColumn names:")
print(student_data.columns.tolist())

Dataset shape: (5000, 12)

Column names:
['student_id', 'cohort_id', 'prediction_date', 'attendance_rate', 'recent_attendance_rate', 'assignments_due', 'assignments_completed', 'assignment_completion_rate', 'missing_assignments', 'late_submissions', 'late_submission_rate', 'days_since_last_submission']


In [31]:
# Verify attendance boundaries

assert student_data["attendance_rate"].between(0, 100).all()
assert student_data["recent_attendance_rate"].between(0, 100).all()


# Verify assignment calculations
assert (student_data["assignments_completed"]+ student_data["missing_assignments"] == student_data["assignments_due"]).all()

# Verify completion rate
calculated_completion = (student_data["assignments_completed"]/ student_data["assignments_due"]) * 100
assert np.allclose(calculated_completion,student_data["assignment_completion_rate"])

# Verify late submissions
assert (student_data["late_submissions"]<= student_data["assignments_completed"]).all()

# Verify late submission boundaries
assert student_data["late_submission_rate"].between(0, 100).all()

# Verify submission recency
assert student_data["days_since_last_submission"].between(0, 30).all()



In [32]:
import plotly.express as px

fig = px.histogram(
    student_data,
    x="attendance_rate",
    nbins=30,
    title="Distribution of Student Attendance",
    labels={
        "attendance_rate": "Attendance Rate (%)",
        "count": "Number of Students"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Attendance Rate (%)",
    yaxis_title="Number of Students"
)

fig.show()

c:\Users\sekar\AppData\Local\Programs\Python\Python311\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [33]:
fig = px.histogram(
    student_data,
    x="assignment_completion_rate",
    nbins=30,
    title="Assignment Completion Distribution",
    labels={
        "assignment_completion_rate": "Completion Rate (%)",
        "count": "Number of Students"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [34]:
fig = px.scatter(
    student_data,
    x="attendance_rate",
    y="assignment_completion_rate",
    opacity=0.5,
    title="Attendance vs Assignment Completion",
    labels={
        "attendance_rate": "Attendance Rate (%)",
        "assignment_completion_rate": "Assignment Completion (%)"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [35]:
correlation = student_data[["attendance_rate","recent_attendance_rate","assignment_completion_rate"]].corr()
display(correlation.round(3))

,attendance_rate,recent_attendance_rate,assignment_completion_rate
attendance_rate,1.000,0.854,0.656
recent_attendance_rate,0.854,1.000,0.563
assignment_completion_rate,0.656,0.563,1.000


In [36]:
assert len(student_data) == n_students
assert len(academic_strength) == n_students
assert len(engagement) == n_students

assert "attendance_rate" in student_data.columns
assert "assignment_completion_rate" in student_data.columns


In [37]:
# Positive trend indicates improvement.
# Negative trend indicates declining performance.
learning_trend = (
    8 * (engagement - 0.5)
    + rng.normal(0, 2.5, n_students)
)
# Generate baseline academic performance
baseline_score = (18 + 72 * academic_strength)
display(
    pd.DataFrame({
        "learning_trend": learning_trend,
        "baseline_score": baseline_score
    }).describe().round(2)
)

,learning_trend,baseline_score
count,5000.00,5000.00
mean,-0.24,52.91
std,3.06,11.78
min,-9.92,19.74
25%,-2.29,44.22
50%,-0.22,52.66
75%,1.84,61.65
max,13.27,85.76


In [38]:
# Assessment 1: Earlier performance
previous_score_1 = (baseline_score- learning_trend+ rng.normal(0, 5, n_students))

# Assessment 2: Middle assessment
previous_score_2 = (baseline_score+ rng.normal(0, 5, n_students))

# Assessment 3: Most recent assessment
previous_score_3 = (baseline_score+ learning_trend+ rng.normal(0, 5, n_students))

# Keep all assessment scores within 0-100
previous_score_1 = np.clip(previous_score_1, 0, 100)
previous_score_2 = np.clip(previous_score_2, 0, 100)
previous_score_3 = np.clip(previous_score_3, 0, 100)

In [39]:
student_data["previous_score_1"] = previous_score_1
student_data["previous_score_2"] = previous_score_2
student_data["previous_score_3"] = previous_score_3

In [40]:
score_columns = ["previous_score_1","previous_score_2","previous_score_3"]
display(student_data[score_columns].describe().round(2))

,previous_score_1,previous_score_2,previous_score_3
count,5000.00,5000.00,5000.00
mean,53.10,52.91,52.64
std,11.64,12.80,14.60
min,18.34,15.57,6.92
25%,44.79,43.66,42.25
50%,52.99,52.61,52.35
75%,61.31,61.83,63.12
max,91.92,93.07,98.13


In [41]:
display(student_data[["student_id"] + score_columns].head(15))

,student_id,previous_score_1,previous_score_2,previous_score_3
0,STU00001,31.109050,40.545986,49.064901
1,STU00002,54.441838,44.553046,38.686174
2,STU00003,55.557314,44.543327,44.151775
3,STU00004,54.629539,50.439654,48.052615
4,STU00005,49.394440,47.326112,57.129738
5,STU00006,37.265783,43.837680,38.329834
6,STU00007,62.660611,57.984082,43.850466
7,STU00008,58.674178,65.410635,73.013077
8,STU00009,45.581208,46.003028,43.065232
9,STU00010,43.308548,60.314149,50.114368


In [42]:
for column in score_columns:
    assert student_data[column].between(0, 100).all()
    assert student_data[column].notna().all()
assert student_data[score_columns].shape == (n_students, 3)



In [43]:
import plotly.express as px

score_distribution = student_data.melt(
    id_vars=["student_id"],
    value_vars=score_columns,
    var_name="Assessment",
    value_name="Score"
)

fig = px.box(
    score_distribution,
    x="Assessment",
    y="Score",
    color="Assessment",
    title="Historical Assessment Score Distribution"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Assessment Score",
    xaxis_title="Historical Assessment"
)

fig.show()

In [44]:
historical_scores = student_data[
    score_columns
].mean().reset_index()

historical_scores.columns = [
    "Assessment",
    "Average Score"
]

fig = px.bar(
    historical_scores,
    x="Assessment",
    y="Average Score",
    text_auto=".2f",
    title="Average Score Across Historical Assessments"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Average Score",
    xaxis_title="Assessment"
)

fig.show()

In [45]:
student_data["average_previous_score"] = (student_data[score_columns].mean(axis=1))
student_data["score_trend"] = (student_data["previous_score_3"]- student_data["previous_score_1"])

In [46]:
# Calculate submission punctuality

punctuality_rate = (100 - student_data["late_submission_rate"])
# Generate assessment variation
assessment_noise = rng.normal(loc=0,scale=6,size=n_students)
# Calculate future assessment score
next_assessment_score = (0.45 * student_data["average_previous_score"]+ 0.20 * student_data["attendance_rate"]+ 0.20 * student_data["assignment_completion_rate"]+ 0.15 * punctuality_rate+ assessment_noise)
# Keep scores within valid boundaries
student_data["next_assessment_score"] = np.clip(next_assessment_score,0,100)

In [47]:
student_data["at_risk_next_assessment"] = (student_data["next_assessment_score"]< risk_threshold).astype(int)

In [48]:
display(student_data[["student_id","average_previous_score","attendance_rate","assignment_completion_rate","next_assessment_score","at_risk_next_assessment"]].head(20).round(2))

,student_id,average_previous_score,attendance_rate,assignment_completion_rate,next_assessment_score,at_risk_next_assessment
0,STU00001,40.24,65.60,52.63,52.30,0
1,STU00002,45.89,59.00,56.52,45.68,0
2,STU00003,48.08,53.93,50.00,60.27,0
3,STU00004,51.04,55.48,56.00,67.49,0
4,STU00005,51.28,81.94,45.45,42.56,0
5,STU00006,39.81,63.30,44.00,60.10,0
6,STU00007,54.83,62.98,48.00,48.43,0
7,STU00008,65.70,74.71,82.35,80.28,0
8,STU00009,44.88,63.55,77.78,65.53,0
9,STU00010,51.25,70.07,55.56,65.23,0


In [49]:
risk_counts = (student_data["at_risk_next_assessment"].value_counts().reindex([0, 1], fill_value=0))
risk_summary = pd.DataFrame({
    "Risk Status": ["Not At Risk", "At Risk"],
    "Students": risk_counts.values
})

risk_summary["Percentage"] = (risk_summary["Students"]/ len(student_data)) * 100
display(risk_summary.round(2))

,Risk Status,Students,Percentage
0,Not At Risk,4725,94.5
1,At Risk,275,5.5


In [50]:
fig = px.bar(
    risk_summary,
    x="Risk Status",
    y="Students",
    text="Percentage",
    title="Student At-Risk Distribution",
    color="Risk Status"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Number of Students"
)
fig.show()

In [51]:
fig = px.scatter(
    student_data,
    x="attendance_rate",
    y="next_assessment_score",
    color="at_risk_next_assessment",
    opacity=0.6,
    title="Attendance vs Next Assessment Score",
    labels={
        "attendance_rate": "Attendance (%)",
        "next_assessment_score": "Next Assessment Score",
        "at_risk_next_assessment": "At Risk"
    }
)
fig.update_layout(
    template="plotly_white"
)
fig.show()

In [52]:
fig = px.scatter(
    student_data,
    x="assignment_completion_rate",
    y="next_assessment_score",
    color="at_risk_next_assessment",
    opacity=0.6,
    title="Assignment Completion vs Next Assessment Score",
    labels={
        "assignment_completion_rate": "Assignment Completion (%)",
        "next_assessment_score": "Next Assessment Score"
    }
)
fig.update_layout(
    template="plotly_white"
)
fig.show()

In [53]:
missingness_plan = {
    "attendance_rate": 0.015,
    "recent_attendance_rate": 0.015,
    "late_submission_rate": 0.020,
    "days_since_last_submission": 0.010
}

In [54]:
for column, missing_rate in missingness_plan.items():

    missing_mask = (rng.random(n_students) < missing_rate)

    student_data.loc[missing_mask,column] = np.nan

In [55]:
missing_summary = pd.DataFrame({
    "Missing Count": student_data.isnull().sum(),
    "Missing Percentage": (
        student_data.isnull().mean() * 100
    )
})

display(missing_summary.round(2))

,Missing Count,Missing Percentage
student_id,0,0.00
cohort_id,0,0.00
prediction_date,0,0.00
attendance_rate,88,1.76
recent_attendance_rate,76,1.52
assignments_due,0,0.00
assignments_completed,0,0.00
assignment_completion_rate,0,0.00
missing_assignments,0,0.00
late_submissions,0,0.00


In [56]:
print("Dataset Shape:", student_data.shape)
print("\nColumn Names:")
print(student_data.columns.tolist())
print("\nDataset Information:")
student_data.info()

Dataset Shape: (5000, 19)

Column Names:
['student_id', 'cohort_id', 'prediction_date', 'attendance_rate', 'recent_attendance_rate', 'assignments_due', 'assignments_completed', 'assignment_completion_rate', 'missing_assignments', 'late_submissions', 'late_submission_rate', 'days_since_last_submission', 'previous_score_1', 'previous_score_2', 'previous_score_3', 'average_previous_score', 'score_trend', 'next_assessment_score', 'at_risk_next_assessment']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   student_id                  5000 non-null   object        
 1   cohort_id                   5000 non-null   object        
 2   prediction_date             5000 non-null   datetime64[ns]
 3   attendance_rate             4912 non-null   float64       
 4   recent_attendance_rate   

In [57]:
print("Dataset Shape:", student_data.shape)
print("\nColumn Names:")
print(student_data.columns.tolist())
print("\nDataset Information:")
student_data.info()

Dataset Shape: (5000, 19)

Column Names:
['student_id', 'cohort_id', 'prediction_date', 'attendance_rate', 'recent_attendance_rate', 'assignments_due', 'assignments_completed', 'assignment_completion_rate', 'missing_assignments', 'late_submissions', 'late_submission_rate', 'days_since_last_submission', 'previous_score_1', 'previous_score_2', 'previous_score_3', 'average_previous_score', 'score_trend', 'next_assessment_score', 'at_risk_next_assessment']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   student_id                  5000 non-null   object        
 1   cohort_id                   5000 non-null   object        
 2   prediction_date             5000 non-null   datetime64[ns]
 3   attendance_rate             4912 non-null   float64       
 4   recent_attendance_rate   

In [58]:
# Student identity
assert student_data["student_id"].is_unique
assert len(student_data) == n_students

# Target validation
assert student_data["at_risk_next_assessment"].notna().all()
assert set(student_data["at_risk_next_assessment"].unique()).issubset({0, 1})

# Future score validation
assert student_data["next_assessment_score"].between(0, 100).all()

# Assessment timeline (if tracked)
if "next_assessment_date" in student_data.columns:
    assert (student_data["prediction_date"] < student_data["next_assessment_date"]).all()

# Required columns
required_columns = [
    "student_id",
    "prediction_date",
    "attendance_rate",
    "assignment_completion_rate",
    "previous_score_1",
    "previous_score_2",
    "previous_score_3",
    "next_assessment_score",
    "at_risk_next_assessment"
]

assert all(column in student_data.columns for column in required_columns)
print("Student dataset validation passed.")


Student dataset validation passed.


In [59]:
from pathlib import Path

current_directory = Path.cwd()
if (current_directory / "data").exists():
    project_root = current_directory
elif (current_directory.parent / "data").exists():
    project_root = current_directory.parent
else:

    raise FileNotFoundError(
        "Cannot locate project data directory. "
        "Run the notebook from the project root or notebooks folder."
    )
output_directory = (
    project_root / "data" / "generated"
)

output_directory.mkdir(
    parents=True,
    exist_ok=True
)


output_path = (
    output_directory / "student_performance.csv"
)


student_data.to_csv(
    output_path,
    index=False
)


print("Dataset successfully saved!")
print("File location:", output_path.resolve())
print("Total records:", len(student_data))
print("Total columns:", len(student_data.columns))

Dataset successfully saved!
File location: D:\student-performance-risk-prediction\data\generated\student_performance.csv
Total records: 5000
Total columns: 19


## Save Visualizations


In [ ]:
# Recreate the three primary visualizations with descriptive variable names
import plotly.express as px

attendance_distribution = px.histogram(
    student_data,
    x="attendance_rate",
    nbins=30,
    title="Distribution of Student Attendance",
    labels={
        "attendance_rate": "Attendance Rate (%)",
        "count": "Number of Students"
    }
)
attendance_distribution.update_layout(
    template="plotly_white",
    xaxis_title="Attendance Rate (%)",
    yaxis_title="Number of Students"
)

assignment_completion_distribution = px.histogram(
    student_data,
    x="assignment_completion_rate",
    nbins=30,
    title="Assignment Completion Distribution",
    labels={
        "assignment_completion_rate": "Completion Rate (%)",
        "count": "Number of Students"
    }
)
assignment_completion_distribution.update_layout(
    template="plotly_white"
)

attendance_vs_assignment = px.scatter(
    student_data,
    x="attendance_rate",
    y="assignment_completion_rate",
    opacity=0.5,
    title="Attendance vs Assignment Completion",
    labels={
        "attendance_rate": "Attendance Rate (%)",
        "assignment_completion_rate": "Assignment Completion (%)"
    }
)
attendance_vs_assignment.update_layout(
    template="plotly_white"
)

# Store figures in a dictionary
figures = {
    "attendance_distribution": attendance_distribution,
    "assignment_completion_distribution": assignment_completion_distribution,
    "attendance_vs_assignment": attendance_vs_assignment
}


In [ ]:
from pathlib import Path

# Locate figures output directory
current_directory = Path.cwd()
if (current_directory / "reports").exists():
    project_root = current_directory
elif (current_directory.parent / "reports").exists():
    project_root = current_directory.parent
else:
    project_root = current_directory

figures_directory = project_root / "reports" / "figures"
figures_directory.mkdir(parents=True, exist_ok=True)


def save_figures(fig_dict, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    saved_files = []
    for name, fig in fig_dict.items():
        json_file = output_dir / f"{name}.json"
        html_file = output_dir / f"{name}.html"
        fig.write_json(str(json_file))
        fig.write_html(str(html_file))
        saved_files.extend([json_file.name, html_file.name])
        
    print(f"Number of saved figures: {len(fig_dict)}")
    print("Saved files:")
    for fname in saved_files:
        print(f"  - {fname}")
        
    expected_files = [
        "attendance_distribution.json",
        "attendance_distribution.html",
        "assignment_completion_distribution.json",
        "assignment_completion_distribution.html",
        "attendance_vs_assignment.json",
        "attendance_vs_assignment.html"
    ]
    
    missing_files = [f for f in expected_files if not (output_dir / f).exists()]
    assert not missing_files, f"Missing expected files: {missing_files}"
    print(f"\nAll {len(expected_files)} expected figure files verified successfully.")


save_figures(figures, figures_directory)
